In [1]:
import torch
import torch.nn as nn
import sys
import os

ROOT = '/home/nicolo_b/Desktop/PhD/RELIABLE_NAS/NOTEBOOK/FAULT_INJECTOR/VITTFI'
sys.path.insert(0, ROOT)
os.chdir(ROOT)

from utils import get_loader

os.chdir(f'{ROOT}/training')

from train_utils import train, test, get_network_untrained, measure_sparsity, setup_loaders, setup_optimizer

In [12]:
MODEL_PATH   = f'{ROOT}/training/trained_models/CIFAR100/MOBILENETV2_X1_0/020_SGD_LR_06_BS_256_EP_200/MOBILENETV2_X1_0_SGD_EP_200_LR_06_CIFAR100.pt'

NETWORK_NAME = 'MOBILENETV2_X1_0'
DATASET      = 'CIFAR100'
DATASET_ROOT = f'{ROOT}/data'   # verifica che sia il path giusto

LR_FINETUNE  = 0.01
FT_EPOCHS    = 50
BATCH_SIZE   = 256
WEIGHT_DECAY = 5e-4
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

In [13]:
device = torch.device(DEVICE)

model = get_network_untrained(NETWORK_NAME, device, DATASET)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)

train_loader, test_loader = setup_loaders(NETWORK_NAME, BATCH_SIZE, DATASET, DATASET_ROOT)

loss_fn = nn.CrossEntropyLoss()
_, start_acc = test(model, device, test_loader, loss_fn)
#start_sparsity = measure_sparsity(model, test_loader, device, threshold=0.0, quantile=0.99)
#print(f'PARTENZA | Test acc: {start_acc:.2f}% | Sparsità: {start_sparsity*100:.2f}%')

/tmp/ipykernel_2365163/3366658336.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=device))


Loading CIFAR100 dataset
Dataset loaded
Batch size:		256 
Number of batches:	40


In [15]:
model.to(device)
optimizer = setup_optimizer('sgd', model, lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)

# LinearLR da 0.07 a 0.001 sulle 50 epoche
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=1.0,           # parte da 0.07 (il lr dell'optimizer)
    end_factor=0.001/LR_FINETUNE,      # arriva a 0.001
    total_iters=FT_EPOCHS
)

for epoch in range(1, FT_EPOCHS + 1):
    train_loss, train_acc, _ = train(model, device, train_loader, optimizer, loss_fn, epoch)
    test_loss, test_acc = test(model, device, test_loader, loss_fn)
    print(f'FT {epoch:02d}/{FT_EPOCHS} | Train: {train_acc:.2f}% | Test: {test_acc:.2f}%')

Epoch 1: 100%|██████████| 196/196 [00:32<00:00,  5.98batch/s, loss=1.1] 


FT 01/50 | Train: 58.04% | Test: 53.61%


Epoch 2: 100%|██████████| 196/196 [00:32<00:00,  5.96batch/s, loss=1.15]


FT 02/50 | Train: 57.88% | Test: 53.85%


Epoch 3: 100%|██████████| 196/196 [00:33<00:00,  5.91batch/s, loss=1.1] 


...

FT 31/50 | Train: 58.23% | Test: 54.14%


Epoch 32: 100%|██████████| 196/196 [00:33<00:00,  5.88batch/s, loss=1.02]


FT 32/50 | Train: 58.37% | Test: 53.26%


Epoch 33:  22%|██▏       | 43/196 [00:07<00:26,  5.72batch/s, loss=1.57]


KeyboardInterrupt: 